В цьому домашньому завданні ми знову працюємо з даними з нашого змагання ["Bank Customer Churn Prediction (DLU Course)"](https://www.kaggle.com/t/7c080c5d8ec64364a93cf4e8f880b6a0).

Тут ми побудуємо рішення задачі класифікації з використанням kNearestNeighboors, знайдемо оптимальні гіперпараметри для цього методу і зробимо базові ансамблі. Це дасть змогу порівняти перформанс моделі з попередніми вивченими методами.

0. Зчитайте дані `train.csv` та зробіть препроцесинг використовуючи написаний Вами скрипт `process_bank_churn.py` так, аби в результаті отримати дані в розбитті X_train, train_targets, X_val, val_targets для експериментів.

  Якщо Вам не вдалось реалізувати в завданні `2.3. Дерева прийняття рішень` скрипт `process_bank_churn.py` - можна скористатись готовим скриптом з запропонованого рішення того завдання.

In [1]:
from pathlib import Path
import sys
import pandas as pd

module_dir = next(Path('..').glob('Дерева*'))
sys.path.insert(0, str(module_dir.resolve()))

from process_bank_churn import preprocess_data

raw_df = pd.read_csv('../data/train.csv')
# kNN порівнює відстані, тому числові ознаки масштабуємо
data = preprocess_data(raw_df, scaler_numeric=True)

X_train = data['train_X']
train_targets = data['train_y']
X_val = data['val_X']
val_targets = data['val_y']

X_train.shape, X_val.shape, train_targets.shape, val_targets.shape

((12000, 13), (3000, 13), (12000,), (3000,))

1. Навчіть на цих даних класифікатор kNN з параметрами за замовченням і виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах. Зробіть заключення про отриману модель: вона хороша/погана, чи є high bias/high variance?

In [2]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score

knn = KNeighborsClassifier()
knn.fit(X_train, train_targets)

train_probs = knn.predict_proba(X_train)[:, 1]
val_probs = knn.predict_proba(X_val)[:, 1]

train_auc = roc_auc_score(train_targets, train_probs)
val_auc = roc_auc_score(val_targets, val_probs)

print(f"ROC AUC (Train): {train_auc:.4f}")
print(f"ROC AUC (Validation): {val_auc:.4f}")
print(f"Різниця train - val: {train_auc - val_auc:.4f}")

ROC AUC (Train): 0.9559
ROC AUC (Validation): 0.8526
Різниця train - val: 0.1034


**Висновок.** Модель з `n_neighbors=5` достатньо хороша на валідації (AUROC ≈ 0.85): вона відділяє клієнтів, які підуть, помітно краще за випадковий рівень 0.5. High bias немає — валідаційний скор не низький.

Є ознака high variance: тренувальний AUROC ≈ 0.96, різниця з валідацією близько 0.10. Частина цього розриву — особливість kNN: під час прогнозу на тренувальних даних точка сама є одним із своїх сусідів, тому train AUROC завищений. Для дерева з попереднього ДЗ розрив був більший (train 1.00, val ≈ 0.77), тож цей kNN узагальнює краще.

2. Використовуючи `GridSearchCV` знайдіть оптимальне значення параметра `n_neighbors` для класифікатора `kNN`. Псотавте крос валідацію на 5 фолдів.

  Після успішного завершення пошуку оптимального гіперпараметра
    - виведіть найкраще значення параметра
    - збережіть в окрему змінну `knn_best` найкращу модель, знайдену з `GridSearchCV`
    - оцініть якість передбачень  `knn_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пукнтом (2) цього завдання? Чи є вона краще за дерево прийняття рішень з попереднього ДЗ?

In [3]:
import time
import numpy as np
from sklearn.model_selection import GridSearchCV

param_grid_knn = {
    'n_neighbors': np.arange(1, 31)
}

knn_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid_knn,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
)

start = time.perf_counter()
knn_search.fit(X_train, train_targets)
knn_search_time = time.perf_counter() - start

print(f'Best n_neighbors: {knn_search.best_params_}')
print(f'Best CV ROC AUC: {knn_search.best_score_:.4f}')
print(f'GridSearchCV time: {knn_search_time:.2f} s')

knn_best = knn_search.best_estimator_

train_probs = knn_best.predict_proba(X_train)[:, 1]
val_probs = knn_best.predict_proba(X_val)[:, 1]

train_auc = roc_auc_score(train_targets, train_probs)
val_auc = roc_auc_score(val_targets, val_probs)

print(f'ROC AUC (Train): {train_auc:.4f}')
print(f'ROC AUC (Validation): {val_auc:.4f}')

Best n_neighbors: {'n_neighbors': 19}
Best CV ROC AUC: 0.8853
GridSearchCV time: 5.83 s
ROC AUC (Train): 0.9227
ROC AUC (Validation): 0.8908


**Висновок.** Найкращий `n_neighbors` — **19** (CV ROC AUC ≈ 0.885). На валідації AUROC **0.8908** проти **0.8526** у kNN з `n_neighbors=5`, тренувальний скор впав з 0.9559 до **0.9227**. Модель стала кращою: менше запам’ятовує шум і краще узагальнює.

До дерева з попереднього ДЗ вона не дотягує. Найкраще дерево, підібране вручну (`max_depth=6`), мало валідаційний AUROC **0.9211**.

3. Виконайте пошук оптимальних гіперпараметрів для `DecisionTreeClassifier` з `GridSearchCV` за сіткою параметрів
  - `max_depth` від 1 до 20 з кроком 2
  - `max_leaf_nodes` від 2 до 10 з кроком 1

  Обовʼязково при цьому ініціюйте модель з фіксацією `random_state`.

  Поставте кросвалідацію на 3 фолди, `scoring='roc_auc'`, та виміряйте, скільки часу потребує пошук оптимальних гіперпараметрів.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення параметра
    - збережіть в окрему змінну `dt_best` найкращу модель, знайдену з `GridSearchCV`
    - оцініть якість передбачень  `dt_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи ця модель краща за ту, що ви знайшли вручну?

In [4]:
from sklearn.tree import DecisionTreeClassifier

param_grid_dt = {
    'max_depth': np.arange(1, 21, 2),
    'max_leaf_nodes': np.arange(2, 11),
}

dt_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid_dt,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
)

start = time.perf_counter()
dt_search.fit(X_train, train_targets)
dt_search_time = time.perf_counter() - start

print(f'Best params: {dt_search.best_params_}')
print(f'Best CV ROC AUC: {dt_search.best_score_:.4f}')
print(f'GridSearchCV time: {dt_search_time:.2f} s')

dt_best = dt_search.best_estimator_

train_probs = dt_best.predict_proba(X_train)[:, 1]
val_probs = dt_best.predict_proba(X_val)[:, 1]

train_auc = roc_auc_score(train_targets, train_probs)
val_auc = roc_auc_score(val_targets, val_probs)

print(f'ROC AUC (Train): {train_auc:.4f}')
print(f'ROC AUC (Validation): {val_auc:.4f}')

Best params: {'max_depth': 5, 'max_leaf_nodes': 10}
Best CV ROC AUC: 0.9014
GridSearchCV time: 0.78 s
ROC AUC (Train): 0.9015
ROC AUC (Validation): 0.9002


**Висновок.** Пошук зайняв менше секунди. Найкращі параметри: `max_depth=5`, `max_leaf_nodes=10` (CV ROC AUC ≈ 0.901). На тренуванні AUROC **0.9015**, на валідації **0.9002** — розриву майже немає, модель не перенавчена.

За дерево, підібране вручну в попередньому ДЗ (`max_depth=6`, val AUROC **0.9211**), ця модель гірша. Сітка обмежує дерево десятьма листками, тому йому не вистачає складності: це ближче до high bias, ніж до high variance.


4. Виконайте пошук оптимальних гіперпараметрів для `DecisionTreeClassifier` з `RandomizedSearchCV` за заданою сіткою параметрів і кількість ітерацій 40.

  Поставте кросвалідацію на 3 фолди, `scoring='roc_auc'`, зафіксуйте `random_seed` процедури крос валідації та виміряйте, скільки часу потребує пошук оптимальних гіперпараметрів.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення параметра
    - збережіть в окрему змінну `dt_random_search_best` найкращу модель, знайдену з `RandomizedSearchCV`
    - оцініть якість передбачень  `dt_random_search_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи ця модель краща за ту, що ви знайшли з `GridSearch`?
    - проаналізуйте параметри `dt_random_search_best` і порівняйте з параметрами `dt_best` - яку бачите відмінність? Ця вправа потрібна аби зрозуміти, як різні налаштування `DecisionTreeClassifier` впливають на якість моделі.

In [5]:
import numpy as np

params_dt = {
    'criterion': ['gini', 'entropy'],
    'splitter': ['best', 'random'],
    'max_depth': np.arange(1, 20),
    'max_leaf_nodes': np.arange(2, 20),
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': [None, 'sqrt', 'log2']
}

In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

dt_random_search = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=42),
    params_dt,
    n_iter=40,
    cv=cv,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
)

start = time.perf_counter()
dt_random_search.fit(X_train, train_targets)
dt_random_search_time = time.perf_counter() - start

print(f'Best params: {dt_random_search.best_params_}')
print(f'Best CV ROC AUC: {dt_random_search.best_score_:.4f}')
print(f'RandomizedSearchCV time: {dt_random_search_time:.2f} s')

dt_random_search_best = dt_random_search.best_estimator_

train_probs = dt_random_search_best.predict_proba(X_train)[:, 1]
val_probs = dt_random_search_best.predict_proba(X_val)[:, 1]

train_auc = roc_auc_score(train_targets, train_probs)
val_auc = roc_auc_score(val_targets, val_probs)

print(f'ROC AUC (Train): {train_auc:.4f}')
print(f'ROC AUC (Validation): {val_auc:.4f}')
print('GridSearch best params:', dt_search.best_params_)

Best params: {'splitter': 'best', 'min_samples_split': 20, 'min_samples_leaf': 2, 'max_leaf_nodes': 14, 'max_features': None, 'max_depth': 16, 'criterion': 'entropy'}
Best CV ROC AUC: 0.9103
RandomizedSearchCV time: 0.24 s
ROC AUC (Train): 0.9169
ROC AUC (Validation): 0.9166
GridSearch best params: {'max_depth': 5, 'max_leaf_nodes': 10}


**Висновок.** Пошук швидший за `GridSearch` (близько 0.3 с проти ~0.9 с): 40 комбінацій замість 90. Найкращі параметри: `criterion='entropy'`, `splitter='best'`, `max_depth=16`, `max_leaf_nodes=14`, `min_samples_split=20`, `min_samples_leaf=2`, `max_features=None`. AUROC train **0.9169**, val **0.9166**.

Модель краща за `dt_best` (val **0.9002**). До ручного дерева з `max_depth=6` (val **0.9211**) ще трохи не дотягує: навіть 14 листків — це менше, ніж дерево глибини 6 без обмеження на кількість листків.

Відмінність параметрів. У `dt_best` спрацювали обмеження сітки: `gini`, `max_depth=5`, `max_leaf_nodes=10`, решта за замовчуванням (`min_samples_split=2`, `min_samples_leaf=1`, `max_features=None`, `splitter='best'`). У випадковому пошуку листя більше (14 замість 10), критерій `entropy`, а `min_samples_split=20` і `min_samples_leaf=2` не дають дробити зовсім дрібні вузли. `max_depth=16` тут майже не обмежує дерево: його зупиняє ліміт у 14 листків. Спільне в обох моделей — `splitter='best'` і `max_features=None`, тобто на кожному розбитті розглядаються всі ознаки. Приріст якості дає саме додаткова ємність (більше листків), а не глибина сама по собі.


5. Якщо у Вас вийшла метрика `AUROC` в цій серії експериментів - зробіть ще один `submission` на Kaggle і додайте код для цього і скріншот скора на публічному лідерборді нижче.

  Сподіваюсь на цьому етапі ви вже відчули себе справжнім дослідником 😉

In [7]:
from process_bank_churn import preprocess_new_data

test_raw_df = pd.read_csv('../data/test.csv')

X_test = preprocess_new_data(
    new_df=test_raw_df,
    numeric_cols=data['numeric_cols'],
    categorical_cols=data['categorical_cols'],
    imputer=data['imputer'],
    encoder=data['encoder'],
    scaler=data['scaler'],
)

test_probs = dt_random_search_best.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'id': test_raw_df['id'],
    'Exited': test_probs,
})

submission.to_csv('submission_dt_random_search.csv', index=False)
submission.head()

,id,Exited
0,15000,0.237911
1,15001,0.012115
2,15002,0.203947
3,15003,0.569848
4,15004,0.082171
